
# LAB 07

#### Swikriti Paudel
#### ACE080BCT061

### TITLE: RNN Encoder-Decoder for Statistical Machine Translation

#### OBJECTIVE:

To implement an RNN Encoder–Decoder model for Statistical Machine Translation (SMT) and understand how recurrent neural networks learn the mapping between source and target language sequences.

### THEORY:

The Encoder–Decoder model is a deep learning architecture designed for Sequence-to-Sequence (Seq2Seq) problems. It is mainly used when both the input and output are sequences but their lengths may differ.

Unlike traditional neural networks that produce a fixed-size output, the Encoder–Decoder model can generate an output sequence one element at a time. This makes it suitable for applications such as:

1) Machine Translation

2) Text Summarization

3) Chatbots

4) Speech Recognition

5) Question Answering

The architecture generally uses Recurrent Neural Networks (RNNs) or Long Short-Term Memory (LSTM) networks because they can process sequential data effectively

### Statistical Machine Translation (SMT)
 It is a machine learning approach that translates text from one language to another using statistical models learned from large bilingual datasets. Traditional SMT systems rely on phrase tables and probability distributions, which often fail to capture the semantic meaning and long-range dependencies between words.

The RNN Encoder–Decoder model was introduced to overcome these limitations by learning continuous vector representations of sentences. It uses two Recurrent Neural Networks (RNNs): an Encoder, which converts the input sentence into a fixed-length vector, and a Decoder, which generates the translated sentence from this vector. This architecture improves translation quality by learning contextual and semantic relationships directly from data.

#### Encoder

The Encoder is the first part of the model.

Its function is to:

1. Read the complete input sequence one element at a time.

2. Learn important information from the sequence.

3. Convert the sequence into a fixed-length vector called the Context Vector.

4. Pass the final hidden state and cell state to the decoder

#### Context Vector

The Context Vector is the compressed representation of the entire input sequence.

It contains:

Semantic information

Sequence information

Learned patterns

The decoder uses this vector to generate the output sequence.

#### Decoder

The Decoder receives the Context Vector from the Encoder.

Its functions are:

1. Start with a special Start-of-Sequence (SOS) token.

2. Predict one output token at a time.

3. Use the previous predicted word to predict the next word.

4. Continue until the End-of-Sequence (EOS) token is generated.

#### Working of Encoder–Decoder Model

The model works in the following steps:

1. Input sequence is given to the Encoder.

2. Encoder processes each input token using LSTM.

3. Final hidden state and cell state become the Context Vector.

4. Decoder receives these states as its initial state.

5. Decoder predicts one output token at each time step.

6. The prediction continues until the End-of-Sequence token is produced.

----------------------------------------

In [41]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.set_default_device(device)
print(f"Using device = {torch.get_default_device()}")

Using device = cpu


## Loading Data



In [42]:
SOS_token = 0 # Start of the Sentence
EOS_token = 1 # End of the Sentence

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

 --> SOS (Start of Sentence) and EOS (End of Sentence) tokens are defined which creates the Lang class to store vocabulary information.

Also it assigns an integer index to each word, counts word frequency.And, 

Creates mappings:

word → index

index → word

In [43]:
# Turn a Unicode string to plain ASCII, thanks to

def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()

This cell cleans the dataset. The following functions are performed here:

Convert Unicode → ASCII

Convert uppercase → lowercase

Remove unnecessary symbols

Only letters and punctuation are kept

In [44]:
def readLangs(path:str):
    lang1 = 'eng'; lang2 = 'fra'
    print("Reading lines...")

    # Read the file and split into lines
    lines = open(path, encoding='utf-8').\
        read().strip().split('\n')

    # Split every line into pairs and normalize (english to french)
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    # Reverse pairs: English-French -> French-English
    pairs = [list(reversed(p)) for p in pairs]

    # Input is French, output is English
    input_lang = Lang(lang2)
    output_lang = Lang(lang1)

    return input_lang, output_lang, pairs

This function loads the translation dataset.

Steps:

1. Read file.
2. Split into lines.
3. Separate English and French sentences.
4. Normalize text.
5. Reverse language pairs (French → English).
6. Create language vocabularies.

The overall purpose is to clean and prepare sentence pairs for training a language model by removing unsuitable examples.


In [45]:
MAX_LENGTH = 5

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

Here, not every sentence is used.

This cell removes long and complex sentences.

Now, let's bundle everything up as per the diagram above: 

### Data Preparation

In [46]:
def prepareData(path):
    input_lang, output_lang, pairs = readLangs(path)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

In [ ]:
PATH = r'data\eng-fra.txt'

input_lang, output_lang, pairs = prepareData(PATH)
print(random.choice(pairs))

output_lang.word2index['am']  # try different English words. 

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
['on nous exploite', 'we re being used']


15

--> the English–French dataset is loaded.

`Note`: Here, we have only inserted lower case in the word2index, therefore, use of uppercase letters will yield an error. 

## Encoder-Decoder Network

### Encoder RNN

In [ ]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.rnn(embedded)
        return output, hidden

Here, we convert the input word indices into dense embedding vectors. During training, these embeddings are learned and gradually capture the semantic relationships and meanings of the words.

Functions:

-Convert word index into embedding vector.

-Process one word after another.

-Produce final hidden state (context vector).

"The encoder does not generate translated words."


### Decoder RNN

In [ ]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden  = self.forward_step(decoder_input, decoder_hidden)
            decoder_outputs.append(decoder_output)

            if target_tensor is not None:
                # Teacher forcing: Feed the target as the next input
                decoder_input = target_tensor[:, i].unsqueeze(1) # Teacher forcing
            else:
                # Without teacher forcing: use its own predictions as the next input
                _, topi = decoder_output.topk(1) # values, index of the highest-scoring word
                decoder_input = topi.squeeze(-1).detach()  # detach from history as input (removes the last dimension before detaching)

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        return decoder_outputs, decoder_hidden, None # We return `None` for consistency in the training loop

    def forward_step(self, input, hidden):
        output = self.embedding(input)
        output = F.relu(output)
        output, hidden = self.rnn(output, hidden)
        output = self.out(output)
        return output, hidden

-> The steps involved are:

1. Receive Encoder hidden state.

2. Start with SOS token.

3. Predict first word.

4. Feed predicted word back.

5. Predict next word.

6. Stop at EOS token.

## Training



## Prepare Training Data

For each pair:
-  we will need an input tensor (indexes of the words in the input sentence) and 
- target tensor (indexes of the words in the target sentence). 
- While creating these vectors we will append the EOS token to both sequences.

In [50]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData(path=PATH)

    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))

    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader

### Training Loop

In [51]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
          decoder_optimizer, criterion):

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor) # using teacher forcing

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

This performs one complete training iteration.

Steps:

1. Read batch.
2. Encoder forward pass.
3. Decoder forward pass.
4. Compute loss.
5. Backpropagation.
6. Update weights using Adam optimizer.

In [52]:
import time
import math

def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

--> Useful for monitoring long training sessions.

In [53]:
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np

def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    # this locator puts ticks at regular intervals
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)

In [54]:
def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
               print_every=100, plot_every=100):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                        epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)

## Evaluation Code

In [55]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn

In [56]:
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate(encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

### Training and Evaluating

In [58]:
hidden_size = 128
batch_size = 32
EPOCHS = 200

input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = DecoderRNN(hidden_size, output_lang.n_words).to(device)

train(train_dataloader, encoder, decoder, EPOCHS, print_every=5, plot_every=5)

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
0m 8s (- 5m 45s) (5 2%) 1.9658
0m 17s (- 5m 33s) (10 5%) 1.2934
0m 26s (- 5m 23s) (15 7%) 1.0896
0m 34s (- 5m 8s) (20 10%) 0.9460
0m 42s (- 4m 54s) (25 12%) 0.8281
0m 50s (- 4m 47s) (30 15%) 0.7256
0m 59s (- 4m 39s) (35 17%) 0.6329
1m 7s (- 4m 31s) (40 20%) 0.5495
1m 16s (- 4m 22s) (45 22%) 0.4786
1m 24s (- 4m 14s) (50 25%) 0.4180
1m 33s (- 4m 7s) (55 27%) 0.3654
1m 42s (- 4m 0s) (60 30%) 0.3215
1m 51s (- 3m 51s) (65 32%) 0.2832
2m 0s (- 3m 44s) (70 35%) 0.2509
2m 9s (- 3m 36s) (75 37%) 0.2183
2m 19s (- 3m 28s) (80 40%) 0.1965
2m 28s (- 3m 20s) (85 42%) 0.1773
2m 37s (- 3m 11s) (90 45%) 0.1585
2m 45s (- 3m 3s) (95 47%) 0.1405
2m 54s (- 2m 54s) (100 50%) 0.1259
3m 2s (- 2m 45s) (105 52%) 0.1141
3m 14s (- 2m 39s) (110 55%) 0.1084
3m 25s (- 2m 32s) (115 57%) 0.0997
3m 39s (- 2m 26s) (120 60%) 0.0928
3m 52s (- 2m 19s) (125 62%) 0.0891
4m 15s (- 2m 17s) (130 65%) 0.08

In [59]:
encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder)

> vous etes beau
= you are beautiful
< you re so pretty <EOS>

> vous etes tres attirante
= you re very attractive
< you re very attractive <EOS>

> vous etes bons
= you are good
< i here so sick <EOS>

> c est un creationiste
= he s a creationist
< he s a creationist <EOS>

> tu es difficile
= you re fussy
< he s after me <EOS>

> je suis mal barre
= i am in trouble
< i am in trouble <EOS>

> je suis tatillonne
= i m fussy
< she is dieting <EOS>

> elles sont toutes mauvaises
= they re all bad
< they re all bad <EOS>

> vous etes merveilleuse
= you re wonderful
< they are melons <EOS>

> nous sommes des hommes
= we are men
< we are men <EOS>



### DISCUSSION 



The RNN Encoder–Decoder model is an important deep learning architecture for statistical machine translation. It consists of two recurrent neural networks: an encoder, which converts the input sentence into a fixed-length context vector, and a decoder, which generates the translated sentence from this context vector. This architecture enables the model to learn the relationship between source and target languages without relying on manually designed translation rules.

During this laboratory experiment, the model demonstrated how sequential data can be effectively processed and translated. The encoder captured the semantic information of the input sequence, while the decoder generated the corresponding output sequence one word at a time. Although the basic RNN Encoder–Decoder performs well for short sentences, its performance may decrease for longer sequences because the fixed-length context vector can lose important information. This limitation led to the development of attention mechanisms, which allow the decoder to focus on different parts of the input sequence during translation.

Overall, the experiment provided practical knowledge of sequence-to-sequence learning and highlighted the significance of neural networks in natural language processing tasks such as machine translation, text summarization, and speech recognition.


## Conclusion

The RNN Encoder–Decoder model is a fundamental sequence-to-sequence architecture used for statistical machine translation. In this laboratory, the model successfully demonstrated how an input sentence can be encoded into a meaningful representation and decoded into a translated output. The experiment improved the understanding of recurrent neural networks, sequence modeling, and encoder–decoder architectures.

Despite its limitations in handling long sequences, the RNN Encoder–Decoder remains a significant milestone in the evolution of neural machine translation. The knowledge gained from this experiment provides a strong foundation for studying advanced models such as attention-based networks and Transformer architectures, which achieve higher translation accuracy and efficiency.
